In [1]:
# PIPELINE POUR NOUVELLE BASE (clients_a_contacter)
import pandas as pd
import joblib
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [2]:

# Charger objets
model = joblib.load("pkl_files/model.pkl")
onehot = joblib.load("pkl_files/onehot.pkl")
ordinal = joblib.load("pkl_files/ordinal.pkl")
minmax_scaler = joblib.load("pkl_files/minmax.pkl")
robust_scaler = joblib.load("pkl_files/robust.pkl")
standard_scaler = joblib.load("pkl_files/standard.pkl")
train_cols = joblib.load("pkl_files/columns_order.pkl")
standard_cols = joblib.load("pkl_files/standard_cols.pkl")
minmax_cols = joblib.load("pkl_files/minmax_cols.pkl")
robust_cols = joblib.load("pkl_files/robust_cols.pkl")


In [3]:

#  Charger nouveaux clients
df_new = pd.read_csv("Data/clients_a_contacter.csv")


In [145]:

# Feature engineering (IDENTIQUE)

df_new["tranche_age"] = pd.cut(
    df_new["age"],
    bins=[0, 30, 40, 50, 60, 70, 80, 100],
    labels=["20-30", "30-40", "40-50", "50-60", "60-70", "70-80", "80+"]
)

df_new["tranche_age_age_vehicule"] = df_new["tranche_age"].astype(str) + "_" + df_new["age_vehicule"].astype(str)
df_new["tranche_age_vehicule_endommage"] = df_new["tranche_age"].astype(str) + "_" + df_new["vehicule_endommage"].astype(str)
df_new["tranche_age_ancien_assure"] = df_new["tranche_age"].astype(str) + "_" + df_new["ancien_assure"].astype(str)
df_new["age_vehicule_vehicule_endommage"] = df_new["age_vehicule"].astype(str) + "_" + df_new["vehicule_endommage"].astype(str)
df_new["age_vehicule_ancien_assure"] = df_new["age_vehicule"].astype(str) + "_" + df_new["ancien_assure"].astype(str)
df_new["vehicule_endommage_ancien_assure"] = df_new["vehicule_endommage"].astype(str) + "_" + df_new["ancien_assure"].astype(str)

In [146]:

# Encodage

onehot_cols = [
    "genre",
    "vehicule_endommage",
    "ancien_assure",
    "tranche_age",
    "tranche_age_age_vehicule",
    "tranche_age_vehicule_endommage",
    "tranche_age_ancien_assure",
    "age_vehicule_vehicule_endommage",
    "age_vehicule_ancien_assure",
    "vehicule_endommage_ancien_assure"
]
encoded = onehot.transform(df_new[onehot_cols])
encoded_df = pd.DataFrame(encoded, columns=onehot.get_feature_names_out(onehot_cols))

df_new["age_vehicule"] = ordinal.transform(df_new[["age_vehicule"]])



In [147]:
# Encodage des variables à forte cardinalité
mean_canal = joblib.load("mean_canal.pkl")
mean_region = joblib.load("mean_region.pkl")
df_new["canal_communication_encoded"] = df_new["canal_communication"].map(mean_canal)
df_new["code_regional_encoded"] = df_new["code_regional"].map(mean_region)

In [149]:
df_new.head()   

,id_client,genre,age,permis_conduire,code_regional,ancien_assure,age_vehicule,vehicule_endommage,prime_annuelle,canal_communication,anciennete,tranche_age,tranche_age_age_vehicule,tranche_age_vehicule_endommage,tranche_age_ancien_assure,age_vehicule_vehicule_endommage,age_vehicule_ancien_assure,vehicule_endommage_ancien_assure,canal_communication_encoded,code_regional_encoded
0,381110,male,25,1,11.0,1,1.0,no,35786.0,152.0,53,20-30,20-30_< 1 an,20-30_no,20-30_1,< 1 an_no,< 1 an_1,no_1,0.028624,0.112760
1,381111,male,40,1,28.0,0,0.0,oui,33762.0,7.0,111,30-40,30-40_1-2 an,30-40_oui,30-40_0,1-2 an_oui,1-2 an_0,oui_0,0.113892,0.187163
2,381112,male,47,1,28.0,0,0.0,oui,40050.0,124.0,199,40-50,40-50_1-2 an,40-50_oui,40-50_0,1-2 an_oui,1-2 an_0,oui_0,0.189148,0.187163
3,381113,male,24,1,27.0,1,1.0,oui,37356.0,152.0,187,20-30,20-30_< 1 an,20-30_oui,20-30_1,< 1 an_oui,< 1 an_1,oui_1,0.028624,0.074035
4,381114,male,27,1,28.0,1,1.0,no,59097.0,152.0,297,20-30,20-30_< 1 an,20-30_no,20-30_1,< 1 an_no,< 1 an_1,no_1,0.028624,0.187163


In [150]:
df_new = pd.concat([df_new.drop(columns=onehot_cols), encoded_df], axis=1)


In [151]:
for col in df_new.columns:
    if df_new[col].dtype == "object":
        print(f"Colonne '{col}' is still object type")
    


In [152]:
df_new.shape

(127037, 77)

In [153]:
df_new.drop(columns=["id_client","code_regional","canal_communication"], inplace=True)

In [154]:
df_new.head()

,age,permis_conduire,age_vehicule,prime_annuelle,anciennete,canal_communication_encoded,code_regional_encoded,genre_male,vehicule_endommage_oui,ancien_assure_1,tranche_age_30-40,tranche_age_40-50,tranche_age_50-60,tranche_age_60-70,tranche_age_70-80,tranche_age_80+,tranche_age_age_vehicule_20-30_< 1 an,tranche_age_age_vehicule_20-30_> 2 ans,tranche_age_age_vehicule_30-40_1-2 an,tranche_age_age_vehicule_30-40_< 1 an,tranche_age_age_vehicule_30-40_> 2 ans,tranche_age_age_vehicule_40-50_1-2 an,tranche_age_age_vehicule_40-50_< 1 an,tranche_age_age_vehicule_40-50_> 2 ans,tranche_age_age_vehicule_50-60_1-2 an,tranche_age_age_vehicule_50-60_< 1 an,tranche_age_age_vehicule_50-60_> 2 ans,tranche_age_age_vehicule_60-70_1-2 an,tranche_age_age_vehicule_60-70_< 1 an,tranche_age_age_vehicule_60-70_> 2 ans,tranche_age_age_vehicule_70-80_1-2 an,tranche_age_age_vehicule_70-80_< 1 an,tranche_age_age_vehicule_70-80_> 2 ans,tranche_age_age_vehicule_80+_1-2 an,tranche_age_age_vehicule_80+_> 2 ans,tranche_age_vehicule_endommage_20-30_oui,tranche_age_vehicule_endommage_30-40_no,tranche_age_vehicule_endommage_30-40_oui,tranche_age_vehicule_endommage_40-50_no,tranche_age_vehicule_endommage_40-50_oui,tranche_age_vehicule_endommage_50-60_no,tranche_age_vehicule_endommage_50-60_oui,tranche_age_vehicule_endommage_60-70_no,tranche_age_vehicule_endommage_60-70_oui,tranche_age_vehicule_endommage_70-80_no,tranche_age_vehicule_endommage_70-80_oui,tranche_age_vehicule_endommage_80+_no,tranche_age_vehicule_endommage_80+_oui,tranche_age_ancien_assure_20-30_1,tranche_age_ancien_assure_30-40_0,tranche_age_ancien_assure_30-40_1,tranche_age_ancien_assure_40-50_0,tranche_age_ancien_assure_40-50_1,tranche_age_ancien_assure_50-60_0,tranche_age_ancien_assure_50-60_1,tranche_age_ancien_assure_60-70_0,tranche_age_ancien_assure_60-70_1,tranche_age_ancien_assure_70-80_0,tranche_age_ancien_assure_70-80_1,tranche_age_ancien_assure_80+_0,tranche_age_ancien_assure_80+_1,age_vehicule_vehicule_endommage_1-2 an_oui,age_vehicule_vehicule_endommage_< 1 an_no,age_vehicule_vehicule_endommage_< 1 an_oui,age_vehicule_vehicule_endommage_> 2 ans_no,age_vehicule_vehicule_endommage_> 2 ans_oui,age_vehicule_ancien_assure_1-2 an_1,age_vehicule_ancien_assure_< 1 an_0,age_vehicule_ancien_assure_< 1 an_1,age_vehicule_ancien_assure_> 2 ans_0,age_vehicule_ancien_assure_> 2 ans_1,vehicule_endommage_ancien_assure_no_1,vehicule_endommage_ancien_assure_oui_0,vehicule_endommage_ancien_assure_oui_1
0,25,1,1.0,35786.0,53,0.028624,0.112760,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,40,1,0.0,33762.0,111,0.113892,0.187163,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,47,1,0.0,40050.0,199,0.189148,0.187163,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,24,1,1.0,37356.0,187,0.028624,0.074035,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,27,1,1.0,59097.0,297,0.028624,0.187163,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0

In [124]:
df_new.shape

(127037, 74)

In [155]:
df_new.drop(columns=["age"], inplace=True)
df_new["age"]=df_first["age"]

In [156]:
df_new.head()

,permis_conduire,age_vehicule,prime_annuelle,anciennete,canal_communication_encoded,code_regional_encoded,genre_male,vehicule_endommage_oui,ancien_assure_1,tranche_age_30-40,tranche_age_40-50,tranche_age_50-60,tranche_age_60-70,tranche_age_70-80,tranche_age_80+,tranche_age_age_vehicule_20-30_< 1 an,tranche_age_age_vehicule_20-30_> 2 ans,tranche_age_age_vehicule_30-40_1-2 an,tranche_age_age_vehicule_30-40_< 1 an,tranche_age_age_vehicule_30-40_> 2 ans,tranche_age_age_vehicule_40-50_1-2 an,tranche_age_age_vehicule_40-50_< 1 an,tranche_age_age_vehicule_40-50_> 2 ans,tranche_age_age_vehicule_50-60_1-2 an,tranche_age_age_vehicule_50-60_< 1 an,tranche_age_age_vehicule_50-60_> 2 ans,tranche_age_age_vehicule_60-70_1-2 an,tranche_age_age_vehicule_60-70_< 1 an,tranche_age_age_vehicule_60-70_> 2 ans,tranche_age_age_vehicule_70-80_1-2 an,tranche_age_age_vehicule_70-80_< 1 an,tranche_age_age_vehicule_70-80_> 2 ans,tranche_age_age_vehicule_80+_1-2 an,tranche_age_age_vehicule_80+_> 2 ans,tranche_age_vehicule_endommage_20-30_oui,tranche_age_vehicule_endommage_30-40_no,tranche_age_vehicule_endommage_30-40_oui,tranche_age_vehicule_endommage_40-50_no,tranche_age_vehicule_endommage_40-50_oui,tranche_age_vehicule_endommage_50-60_no,tranche_age_vehicule_endommage_50-60_oui,tranche_age_vehicule_endommage_60-70_no,tranche_age_vehicule_endommage_60-70_oui,tranche_age_vehicule_endommage_70-80_no,tranche_age_vehicule_endommage_70-80_oui,tranche_age_vehicule_endommage_80+_no,tranche_age_vehicule_endommage_80+_oui,tranche_age_ancien_assure_20-30_1,tranche_age_ancien_assure_30-40_0,tranche_age_ancien_assure_30-40_1,tranche_age_ancien_assure_40-50_0,tranche_age_ancien_assure_40-50_1,tranche_age_ancien_assure_50-60_0,tranche_age_ancien_assure_50-60_1,tranche_age_ancien_assure_60-70_0,tranche_age_ancien_assure_60-70_1,tranche_age_ancien_assure_70-80_0,tranche_age_ancien_assure_70-80_1,tranche_age_ancien_assure_80+_0,tranche_age_ancien_assure_80+_1,age_vehicule_vehicule_endommage_1-2 an_oui,age_vehicule_vehicule_endommage_< 1 an_no,age_vehicule_vehicule_endommage_< 1 an_oui,age_vehicule_vehicule_endommage_> 2 ans_no,age_vehicule_vehicule_endommage_> 2 ans_oui,age_vehicule_ancien_assure_1-2 an_1,age_vehicule_ancien_assure_< 1 an_0,age_vehicule_ancien_assure_< 1 an_1,age_vehicule_ancien_assure_> 2 ans_0,age_vehicule_ancien_assure_> 2 ans_1,vehicule_endommage_ancien_assure_no_1,vehicule_endommage_ancien_assure_oui_0,vehicule_endommage_ancien_assure_oui_1,age
0,1,1.0,35786.0,53,0.028624,0.112760,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,25
1,1,0.0,33762.0,111,0.113892,0.187163,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,40
2,1,0.0,40050.0,199,0.189148,0.187163,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,47
3,1,1.0,37356.0,187,0.028624,0.074035,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,24
4,1,1.0,59097.0,297,0.028624,0.187163,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.

In [126]:
df_new.drop(columns=["age"], inplace=True)
df_first = pd.read_csv("Data/clients_a_contacter.csv")
df_new["age"] = df_first["age"]

In [157]:
df_new.head()

,permis_conduire,age_vehicule,prime_annuelle,anciennete,canal_communication_encoded,code_regional_encoded,genre_male,vehicule_endommage_oui,ancien_assure_1,tranche_age_30-40,tranche_age_40-50,tranche_age_50-60,tranche_age_60-70,tranche_age_70-80,tranche_age_80+,tranche_age_age_vehicule_20-30_< 1 an,tranche_age_age_vehicule_20-30_> 2 ans,tranche_age_age_vehicule_30-40_1-2 an,tranche_age_age_vehicule_30-40_< 1 an,tranche_age_age_vehicule_30-40_> 2 ans,tranche_age_age_vehicule_40-50_1-2 an,tranche_age_age_vehicule_40-50_< 1 an,tranche_age_age_vehicule_40-50_> 2 ans,tranche_age_age_vehicule_50-60_1-2 an,tranche_age_age_vehicule_50-60_< 1 an,tranche_age_age_vehicule_50-60_> 2 ans,tranche_age_age_vehicule_60-70_1-2 an,tranche_age_age_vehicule_60-70_< 1 an,tranche_age_age_vehicule_60-70_> 2 ans,tranche_age_age_vehicule_70-80_1-2 an,tranche_age_age_vehicule_70-80_< 1 an,tranche_age_age_vehicule_70-80_> 2 ans,tranche_age_age_vehicule_80+_1-2 an,tranche_age_age_vehicule_80+_> 2 ans,tranche_age_vehicule_endommage_20-30_oui,tranche_age_vehicule_endommage_30-40_no,tranche_age_vehicule_endommage_30-40_oui,tranche_age_vehicule_endommage_40-50_no,tranche_age_vehicule_endommage_40-50_oui,tranche_age_vehicule_endommage_50-60_no,tranche_age_vehicule_endommage_50-60_oui,tranche_age_vehicule_endommage_60-70_no,tranche_age_vehicule_endommage_60-70_oui,tranche_age_vehicule_endommage_70-80_no,tranche_age_vehicule_endommage_70-80_oui,tranche_age_vehicule_endommage_80+_no,tranche_age_vehicule_endommage_80+_oui,tranche_age_ancien_assure_20-30_1,tranche_age_ancien_assure_30-40_0,tranche_age_ancien_assure_30-40_1,tranche_age_ancien_assure_40-50_0,tranche_age_ancien_assure_40-50_1,tranche_age_ancien_assure_50-60_0,tranche_age_ancien_assure_50-60_1,tranche_age_ancien_assure_60-70_0,tranche_age_ancien_assure_60-70_1,tranche_age_ancien_assure_70-80_0,tranche_age_ancien_assure_70-80_1,tranche_age_ancien_assure_80+_0,tranche_age_ancien_assure_80+_1,age_vehicule_vehicule_endommage_1-2 an_oui,age_vehicule_vehicule_endommage_< 1 an_no,age_vehicule_vehicule_endommage_< 1 an_oui,age_vehicule_vehicule_endommage_> 2 ans_no,age_vehicule_vehicule_endommage_> 2 ans_oui,age_vehicule_ancien_assure_1-2 an_1,age_vehicule_ancien_assure_< 1 an_0,age_vehicule_ancien_assure_< 1 an_1,age_vehicule_ancien_assure_> 2 ans_0,age_vehicule_ancien_assure_> 2 ans_1,vehicule_endommage_ancien_assure_no_1,vehicule_endommage_ancien_assure_oui_0,vehicule_endommage_ancien_assure_oui_1,age
0,1,1.0,35786.0,53,0.028624,0.112760,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,25
1,1,0.0,33762.0,111,0.113892,0.187163,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,40
2,1,0.0,40050.0,199,0.189148,0.187163,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,47
3,1,1.0,37356.0,187,0.028624,0.074035,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,24
4,1,1.0,59097.0,297,0.028624,0.187163,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.

In [158]:
# # alignement
# df_new = df_new.reindex(columns=train_cols, fill_value=0)

# # 🔥 recalcul correct
# standard_cols = [
#     col for col in train_cols
#     if col not in minmax_cols + robust_cols
# ]

minmax_cols = ["age", "permis_conduire"]

# Colonnes pour RobustScaler (avec outliers)
robust_cols = ["prime_annuelle"]

# Colonnes restantes → StandardScaler
# On enlève celles déjà utilisées
standard_cols = [
    col for col in df_new.columns
    if col not in minmax_cols + robust_cols
]

# scaling
df_new[minmax_cols] = minmax_scaler.transform(df_new[minmax_cols])
df_new[robust_cols] = robust_scaler.transform(df_new[robust_cols])
df_new[standard_cols] = standard_scaler.transform(df_new[standard_cols])

In [159]:
# Nettoyer noms colonnes
df_new.columns = df_new.columns.str.replace('[\\[\\]<>]', '_', regex=True)

In [164]:
import xgboost as xgb

dnew = xgb.DMatrix(df_new)
proba = model.predict(dnew)

In [165]:
# Sélection marketing
df_new["proba"] = proba
df_new["prediction"] = (df_new["proba"] > 0.7).astype(int)

In [167]:
# Sauvegarde
clients_cibles = df_new[(df_new["proba"] >= 0.3) & (df_new["proba"] <= 0.7)]
clients_cibles.to_csv("clients_cibles.csv", index=False)

In [168]:
clients_cibles.head()

,permis_conduire,age_vehicule,prime_annuelle,anciennete,canal_communication_encoded,code_regional_encoded,genre_male,vehicule_endommage_oui,ancien_assure_1,tranche_age_30-40,tranche_age_40-50,tranche_age_50-60,tranche_age_60-70,tranche_age_70-80,tranche_age_80+,tranche_age_age_vehicule_20-30__ 1 an,tranche_age_age_vehicule_20-30__ 2 ans,tranche_age_age_vehicule_30-40_1-2 an,tranche_age_age_vehicule_30-40__ 1 an,tranche_age_age_vehicule_30-40__ 2 ans,tranche_age_age_vehicule_40-50_1-2 an,tranche_age_age_vehicule_40-50__ 1 an,tranche_age_age_vehicule_40-50__ 2 ans,tranche_age_age_vehicule_50-60_1-2 an,tranche_age_age_vehicule_50-60__ 1 an,tranche_age_age_vehicule_50-60__ 2 ans,tranche_age_age_vehicule_60-70_1-2 an,tranche_age_age_vehicule_60-70__ 1 an,tranche_age_age_vehicule_60-70__ 2 ans,tranche_age_age_vehicule_70-80_1-2 an,tranche_age_age_vehicule_70-80__ 1 an,tranche_age_age_vehicule_70-80__ 2 ans,tranche_age_age_vehicule_80+_1-2 an,tranche_age_age_vehicule_80+__ 2 ans,tranche_age_vehicule_endommage_20-30_oui,tranche_age_vehicule_endommage_30-40_no,tranche_age_vehicule_endommage_30-40_oui,tranche_age_vehicule_endommage_40-50_no,tranche_age_vehicule_endommage_40-50_oui,tranche_age_vehicule_endommage_50-60_no,tranche_age_vehicule_endommage_50-60_oui,tranche_age_vehicule_endommage_60-70_no,tranche_age_vehicule_endommage_60-70_oui,tranche_age_vehicule_endommage_70-80_no,tranche_age_vehicule_endommage_70-80_oui,tranche_age_vehicule_endommage_80+_no,tranche_age_vehicule_endommage_80+_oui,tranche_age_ancien_assure_20-30_1,tranche_age_ancien_assure_30-40_0,tranche_age_ancien_assure_30-40_1,tranche_age_ancien_assure_40-50_0,tranche_age_ancien_assure_40-50_1,tranche_age_ancien_assure_50-60_0,tranche_age_ancien_assure_50-60_1,tranche_age_ancien_assure_60-70_0,tranche_age_ancien_assure_60-70_1,tranche_age_ancien_assure_70-80_0,tranche_age_ancien_assure_70-80_1,tranche_age_ancien_assure_80+_0,tranche_age_ancien_assure_80+_1,age_vehicule_vehicule_endommage_1-2 an_oui,age_vehicule_vehicule_endommage__ 1 an_no,age_vehicule_vehicule_endommage__ 1 an_oui,age_vehicule_vehicule_endommage__ 2 ans_no,age_vehicule_vehicule_endommage__ 2 ans_oui,age_vehicule_ancien_assure_1-2 an_1,age_vehicule_ancien_assure__ 1 an_0,age_vehicule_ancien_assure__ 1 an_1,age_vehicule_ancien_assure__ 2 ans_0,age_vehicule_ancien_assure__ 2 ans_1,vehicule_endommage_ancien_assure_no_1,vehicule_endommage_ancien_assure_oui_0,vehicule_endommage_ancien_assure_oui_1,age,proba,prediction
11,1.0,2.567308,0.025054,1.192309,0.769349,1.425461,0.921850,0.99025,-0.91994,-0.411413,-0.499424,-0.367331,3.552827,-0.204033,-0.016797,-0.821282,-0.01403,-0.358971,-0.167929,-0.056495,-0.477687,-0.029383,-0.11467,-0.344391,-0.022334,-0.111878,-0.262298,-0.014146,10.591157,-0.191432,-0.006007,-0.067822,-0.015997,-0.005122,-0.384949,-0.244054,-0.311683,-0.257581,-0.399126,-0.19965,-0.296000,-0.169092,4.574369,-0.129032,-0.155432,-0.012548,-0.011165,-0.618542,-0.324147,-0.228636,-0.412121,-0.239961,-0.301995,-0.191322,4.503233,-0.164711,-0.157893,-0.126057,-0.011017,-0.012678,-0.712117,-0.66331,-0.380183,-0.005122,4.768019,-0.455461,-0.412442,-0.633601,4.773258,-0.010715,-0.872737,1.043166,-0.163435,0.676923,0.687061,0
14,1.0,-0.893030,-1.935366,0.176076,0.769349,-0.457089,-1.084776,0.99025,-0.91994,-0.411413,-0.499424,2.722341,-0.281466,-0.204033,-0.016797,-0.821282,-0.01403,-0.358971,-0.167929,-0.056495,-0.477687,-0.029383,-0.11467,2.903677,-0.022334,-0.111878,-0.262298,-0.014146,-0.094418,-0.191432,-0.006007,-0.067822,-0.015997,-0.005122,-0.384949,-0.244054,-0.311683,-0.257581,-0.399126,-0.19965,3.378381,-0.169092,-0.218609,-0.129032,-0.155432,-0.012548,-0.011165,-0.618542,-0.324147,-0.228636,-0.412121,-0.239961,3.311314,-0.191322,-0.222063,-0.164711,-0.157893,-0.126057,-0.011017,-0.012678,1.404263,-0.66331,-0.380183,-0.005122,-0.209731,-0.455461,-0.412442,-0.633601,-0.209501,-0.010715,-0.872737,1.043166,-0.163435,0.538462,0.627874,0
16,1.0,0.837139,0.731967,0.630392,-1.088766,0.419769,0.921

In [169]:
df_new["proba"].describe()

count    127037.000000
mean          0.333196
std           0.339291
min           0.000324
25%           0.002528
50%           0.193462
75%           0.714684
max           0.903390
Name: proba, dtype: float64

In [171]:
df_new["proba"].head()

0    0.004719
1    0.757457
2    0.766838
3    0.032191
4    0.002722
Name: proba, dtype: float32

In [172]:
df_original = pd.read_csv("Data/clients_a_contacter.csv")

df_original["proba"] = df_new["proba"]
df_original["prediction"] = df_new["prediction"]

In [173]:
clients_cibles = df_original[
    (df_original["proba"] > 0.3) & (df_original["proba"] < 0.7)
]

In [ ]:
clients_cibles.to_csv("Data/clients_cibles.csv", index=False)

In [175]:
clients_cibles.head()

,id_client,genre,age,permis_conduire,code_regional,ancien_assure,age_vehicule,vehicule_endommage,prime_annuelle,canal_communication,anciennete,proba,prediction
11,381121,male,64,1,28.0,0,> 2 ans,oui,32051.0,124.0,254,0.687061,0
14,381124,femelle,55,1,48.0,0,1-2 an,oui,2630.0,124.0,169,0.627874,0
16,381126,male,25,1,24.0,0,< 1 an,oui,42660.0,152.0,207,0.630210,0
24,381134,male,58,1,28.0,0,1-2 an,oui,41016.0,24.0,201,0.627921,0
30,381140,femelle,70,1,8.0,0,1-2 an,oui,36804.0,124.0,256,0.475750,0


In [176]:
clients_cibles["prediction"].value_counts()

prediction
0    25289
Name: count, dtype: int64